In [1]:
import sys

sys.path.insert(0, "/opt/project")

from src.spark_session import create_spark_session

spark = create_spark_session(
    "FastFood-MinIO-Validation"
)

sc = spark.sparkContext

https://repos.spark-packages.org/ added as a remote repository with the name: repo-1
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9f6534ac-4817-42bb-8a04-d4933fe6e4f5;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (2096ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (365331ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-openssl-1.0.7.Final.jar ...
	[SUCCESSFUL ] org.wildfly.openssl#wildfly-openssl;1.0.7.Final!wildfly-openssl.jar (1671ms)
:: resolution report :: resolve 17347ms :: artifacts dl 369106ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [

In [2]:
data = [
    ("Casablanca", 100.0),
    ("Rabat", 150.0),
    ("Casablanca", 200.0),
    ("Tanger", 180.0)
]

df = spark.createDataFrame(
    data,
    ["city", "amount_mad"]
)

df.show()

+----------+----------+
|      city|amount_mad|
+----------+----------+
|Casablanca|     100.0|
|     Rabat|     150.0|
|Casablanca|     200.0|
|    Tanger|     180.0|
+----------+----------+



In [3]:
test_path = "s3a://lake/tests/cluster_validation/"

(
    df.write
    .mode("overwrite")
    .parquet(test_path)
)

print("WRITE SUCCESSFUL")

26/11/21 21:11:01 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


WRITE SUCCESSFUL


In [4]:
loaded_df = spark.read.parquet(test_path)

loaded_df.show()

assert loaded_df.count() == 4

print("READ SUCCESSFUL")

+----------+----------+
|      city|amount_mad|
+----------+----------+
|Casablanca|     100.0|
|Casablanca|     200.0|
|    Tanger|     180.0|
|     Rabat|     150.0|
+----------+----------+

READ SUCCESSFUL


In [5]:
from pyspark.sql import functions as F

result = (
    loaded_df
    .groupBy("city")
    .agg(
        F.sum("amount_mad").alias("revenue")
    )
    .orderBy("city")
)

result.show()

+----------+-------+
|      city|revenue|
+----------+-------+
|Casablanca|  300.0|
|     Rabat|  150.0|
|    Tanger|  180.0|
+----------+-------+



26/11/22 03:42:26 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 11008343 ms exceeds timeout 120000 ms
26/11/22 03:42:27 ERROR TaskSchedulerImpl: Lost executor 0 on 172.29.0.5: Executor heartbeat timed out after 11008343 ms


In [6]:
assert loaded_df.count() == 4

assert result.count() == 3

assert loaded_df.agg(
    F.sum("amount_mad")
).first()[0] == 630.0

print("=" * 45)
print("SPARK + MINIO : ALL TESTS PASSED")
print("=" * 45)

SPARK + MINIO : ALL TESTS PASSED


In [7]:
spark.stop()